In [ ]:
!pip install transformers datasets accelerate

In [ ]:
!pip install -U datasets

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving emotion-emotion_69k.csv to emotion-emotion_69k (1).csv


In [ ]:
import pandas as pd

data = pd.read_csv("/content/emotion-emotion_69k.csv")
data.head()

,Unnamed: 0,Situation,emotion,empathetic_dialogues,labels,Unnamed: 5,Unnamed: 6
0,0,I remember going to the fireworks with my best...,sentimental,Customer :I remember going to see the firework...,"Was this a friend you were in love with, or ju...",NaN,NaN
1,1,I remember going to the fireworks with my best...,sentimental,Customer :This was a best friend. I miss her.\...,Where has she gone?,NaN,NaN
2,2,I remember going to the fireworks with my best...,sentimental,Customer :We no longer talk.\nAgent :,Oh was this something that happened because of...,NaN,NaN
3,3,I remember going to the fireworks with my best...,sentimental,Customer :Was this a friend you were in love w...,This was a best friend. I miss her.,NaN,NaN
4,4,I remember going to the fireworks with my best...,sentimental,Customer :Where has she gone?\nAgent :,We no longer talk.,NaN,NaN


In [ ]:
data.columns

Index(['Unnamed: 0', 'Situation', 'emotion', 'empathetic_dialogues', 'labels',
       'Unnamed: 5', 'Unnamed: 6'],
      dtype='object')

In [ ]:
# 'Situation' is the user's statement/context
data = data[['Situation', 'empathetic_dialogues']]

# Rename columns for clarity
data = data.rename(columns={'Situation': 'prompt', 'empathetic_dialogues': 'utterance'})

data = data.sample(3000, random_state=42)

data.head()

,prompt,utterance
37054,My husband and I have been married 22 years. ...,"Customer :No, just married kinda young, but we..."
15618,I can't wait to get to the ocean!,"Customer :No, I've never done that but my favo..."
44316,I just knew I was going to pass my drivers tes...,Customer :It sounds like you were very prepare...
10562,I almost make a list before I go food shopping...,Customer :I always make a list before I go foo...
16123,I lost my car keys at a bar last night.,Customer :Oh my god! Well please get ALL your ...


In [ ]:
data["text"] = data["prompt"] + " " + data["utterance"]
data = data[["text"]]

In [ ]:
# Step 4: Convert to Hugging Face Dataset
from datasets import Dataset

dataset = Dataset.from_pandas(data)
dataset

Dataset({
    features: ['text', '__index_level_0__'],
    num_rows: 3000
})

In [ ]:
# Step 5: Load tokenizer
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Step 6: Tokenize dataset
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

tokenized_dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [ ]:
# 5️⃣ Add labels
def add_labels(example):
    example["labels"] = example["input_ids"]
    return example

tokenized_dataset = tokenized_dataset.map(add_labels)


Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [ ]:
# Step 7: Load model
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("distilgpt2")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    num_train_epochs=1,
    max_steps=200
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

In [ ]:
# 9️⃣ Train
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=200, training_loss=2.0317234802246094, metrics={'train_runtime': 899.9348, 'train_samples_per_second': 1.778, 'train_steps_per_second': 0.222, 'total_flos': 26129675059200.0, 'train_loss': 2.0317234802246094, 'epoch': 0.5333333333333333})

In [ ]:
model.save_pretrained("/content/drive/MyDrive/mental_model")
tokenizer.save_pretrained("/content/drive/MyDrive/mental_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/mental_model/tokenizer_config.json',
 '/content/drive/MyDrive/mental_model/tokenizer.json')